In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

# FLenQA J-Lens drift intervention

This experiment asks whether the layer-2 drift between matched short and long prompts is related to the answer prediction. The intervention changes only selected J-Lens coefficients at the explicitly chosen `final_prompt` residual position.

In [ ]:
import string

import jlens
import pandas as pd
import torch
import transformers
from datasets import load_from_disk
from jlens.hooks import ActivationRecorder

from experiments.jlens_readout_sanity.constants import LENS_PATH, MODEL_PATH
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows, prepare_prompts
from jlens_reasoning.benchmarks.flenqa.lens import ApplyLensRunner, LensRunners, run_prompt
from jlens_reasoning.benchmarks.flenqa.positions import prepare_prompt
from jlens_reasoning.evaluation import evaluate_next_token
from jlens_reasoning.experiments_utils.interventions import (
    LensCoordinatePatcher,
    coordinate_patch,
    jlens_vector,
    lens_coordinates,
)
from jlens_reasoning.experiments_utils.validation import validate_model_lens

LAYER = 2
TOP_K = 250
SHORT_CTX_SIZE, LONG_CTX_SIZE = 250, 1000
N_PROBLEMS = None
K_VALUES = (10, 25, 50)
ALPHAS = (0.0, 0.5, 1.0)
POSITION_LABEL = "final_prompt"
MAX_SEQ_LEN = 4096

In [ ]:
dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["eval"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows)

causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
causal_lm.eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
validate_model_lens(model, lens)
assert LAYER in lens.source_layers
runners = LensRunners(
    ApplyLensRunner(lens, model, True, layers=(LAYER,)),
    ApplyLensRunner(lens, model, False, layers=(LAYER,)),
)
unembedding_weight = causal_lm.get_output_embeddings().weight
print(f"model={MODEL_PATH}, layer={LAYER}, top_k={TOP_K}")

## Load matched FLenQA prompts

We select all problem IDs that have both nominal context lengths. Each pair is prepared and tokenized independently; no short sequence is padded or aligned to the long sequence.

In [ ]:
by_problem = {}
for row in rows:
    by_problem.setdefault(row.problem_id, []).append(row)
problem_ids = [
    problem_id
    for problem_id, problem_rows in by_problem.items()
    if {row.ctx_size_declared for row in problem_rows} >= {SHORT_CTX_SIZE, LONG_CTX_SIZE}
][:N_PROBLEMS]
selected_rows = [row for row in rows if row.problem_id in problem_ids]
prepared_prompts = prepare_prompts(selected_rows)

def nominal_context_size(prompt):
    sizes = {item.ctx_size for item in prompt.provenance}
    if len(sizes) != 1:
        raise ValueError(f"Prompt {prompt.prompt_id} spans context sizes {sizes}")
    return sizes.pop()

pairs = []
for problem_id in problem_ids:
    candidates = [p for p in prepared_prompts if p.problem_id == problem_id]
    short = sorted((p for p in candidates if nominal_context_size(p) == SHORT_CTX_SIZE), key=lambda p: p.canonical_index)[0]
    long = sorted((p for p in candidates if nominal_context_size(p) == LONG_CTX_SIZE), key=lambda p: p.canonical_index)[0]
    if (short.task, short.label) != (long.task, long.label):
        raise ValueError(f"Matched prompts disagree on task/label for problem {problem_id}")
    pairs.append({"problem_id": problem_id, "short": short, "long": long})

pair_info = pd.DataFrame([
    {
        "problem_id": pair["problem_id"],
        "label": pair["short"].label,
        "short_prompt_id": pair["short"].prompt_id,
        "long_prompt_id": pair["long"].prompt_id,
        "short_prompt": pair["short"].text,
        "long_prompt": pair["long"].text,
    }
    for pair in pairs
])
display(pair_info)

## Measure layer-2 J-Lens drift

The existing paired lens runner computes the top-250 Jacobian-Lens rows at one explicit position per prompt. Here drift is the absolute change in reciprocal-rank prominence; a token missing from one top-250 list has prominence zero.

In [ ]:
prepared_by_id = {}
topk_tables = []
for pair in pairs:
    for prompt in (pair["short"], pair["long"]):
        prepared = prepare_prompt(prompt, tokenizer, max_seq_len=MAX_SEQ_LEN)
        position = prepared.positions[POSITION_LABEL][0]
        prepared_by_id[prompt.prompt_id] = (prepared, position)
        result = run_prompt(
            prepared, runners=runners, top_k=TOP_K, max_seq_len=MAX_SEQ_LEN,
            logits_rtol=1e-5, logits_atol=1e-6,
        )
        topk_tables.append(result.batches["topk"].to_pandas())

topk = pd.concat(topk_tables, ignore_index=True)
topk = topk.query("lens_kind == 'jacobian' and layer == @LAYER").copy()
topk["rank_score"] = 1 / topk["rank"]
drift = topk.pivot_table(
    index="token_id", columns="prompt_id", values="rank_score", fill_value=0
)
drift["short_score"] = drift[[pair["short"].prompt_id for pair in pairs]].mean(axis=1)
drift["long_score"] = drift[[pair["long"].prompt_id for pair in pairs]].mean(axis=1)
drift["drift"] = (drift["short_score"] - drift["long_score"]).abs()
drift.index.name = "token_id"
drift = drift.reset_index().sort_values("drift", ascending=False)

token_text = {
    int(token_id): tokenizer.decode([int(token_id)], clean_up_tokenization_spaces=False)
    for token_id in drift["token_id"]
}
drift["token"] = drift["token_id"].map(token_text)
display(drift.head(20))

In [ ]:
CONTROL_SURFACES = {"yes", "no", "true", "false", "answer", "response", "think", "/think"}
SPECIAL_SURFACES = {text.casefold() for text in tokenizer.all_special_tokens}

def exclusion_reason(token_id, token):
    normalized = token.strip().casefold().lstrip("▁Ġ")
    if token_id in set(tokenizer.all_special_ids) or normalized in SPECIAL_SURFACES:
        return "special token"
    if not normalized or all(character in string.punctuation for character in normalized):
        return "formatting/empty"
    if normalized in CONTROL_SURFACES or normalized.strip(string.punctuation) in CONTROL_SURFACES:
        return "answer/control surface"
    return None

drift["excluded_reason"] = [
    exclusion_reason(int(row.token_id), row.token)
    for row in drift.itertuples()
]
filtered_out = drift[drift["excluded_reason"].notna()]
retained = drift[drift["excluded_reason"].isna()].copy()
print("Filtered concepts:")
display(filtered_out[["token_id", "token", "drift", "excluded_reason"]].head(30))
print("Retained concepts used for directions:")
display(retained[["token_id", "token", "short_score", "long_score", "drift"]].head(max(K_VALUES)))

## Build intervention directions

Each retained token becomes a J-Lens vector from its unembedding row and the fitted layer-2 Jacobian. Coefficients are computed independently for each prompt's own activation tensor and own selected position.

In [ ]:
def activation_for_prompt(prompt_id):
    prepared, position = prepared_by_id[prompt_id]
    input_ids = torch.tensor([prepared.input_ids], device=context.device)
    with torch.inference_mode(), ActivationRecorder(model.layers, at=(LAYER,)) as recorder:
        causal_lm(input_ids=input_ids, use_cache=False)
    activation = recorder.activations[LAYER].detach()
    assert activation.shape[1] == len(prepared.input_ids)
    return input_ids, position, activation

activation_by_id = {}
for pair in pairs:
    for prompt in (pair["short"], pair["long"]):
        activation_by_id[prompt.prompt_id] = activation_for_prompt(prompt.prompt_id)

selected_token_ids = retained.head(max(K_VALUES))["token_id"].astype(int).tolist()
vectors = torch.stack([
    jlens_vector(lens, unembedding_weight, layer=LAYER, token_id=token_id)
    for token_id in selected_token_ids
]).T
print("selected token IDs/tokens:")
display(pd.DataFrame({"token_id": selected_token_ids, "token": [token_text[i] for i in selected_token_ids]}))
print("direction matrix shape:", tuple(vectors.shape))

In [ ]:
coefficient_rows = []
for pair in pairs:
    short_id, long_id = pair["short"].prompt_id, pair["long"].prompt_id
    short_activation = activation_by_id[short_id][2]
    long_activation = activation_by_id[long_id][2]
    short_position, long_position = activation_by_id[short_id][1], activation_by_id[long_id][1]
    short_coefficients = lens_coordinates(short_activation[:, short_position, :], vectors)[0].cpu()
    long_coefficients = lens_coordinates(long_activation[:, long_position, :], vectors)[0].cpu()
    coefficient_rows.extend({
        "problem_id": pair["problem_id"], "token": token_text[token_id],
        "short_coefficient": float(short_coefficients[index]),
        "long_coefficient": float(long_coefficients[index]),
        "change_long_minus_short": float(long_coefficients[index] - short_coefficients[index]),
    } for index, token_id in enumerate(selected_token_ids))

coefficients = pd.DataFrame(coefficient_rows)
display(coefficients)

# This is the same coefficient-level operation used by LensCoordinatePatcher.
probe_source = activation_by_id[pairs[0]["long"].prompt_id][2][:, activation_by_id[pairs[0]["long"].prompt_id][1], :]
probe_target = activation_by_id[pairs[0]["short"].prompt_id][2][:, activation_by_id[pairs[0]["short"].prompt_id][1], :]
probe = coordinate_patch(probe_source, vectors, lens_coordinates(probe_target, vectors), alpha=1.0)
print("probe coefficient shift:", (lens_coordinates(probe, vectors) - lens_coordinates(probe_source, vectors)).abs().mean().item())

## Run both coefficient interventions

The hook receives the full activation sequence for the current prompt. We copy the source prompt's own coefficients for every position, then replace only the explicitly selected position with the other prompt's coefficients. This avoids aligning or mixing different-length sequences.

In [ ]:
def next_token_score(logits, prompt):
    expected = "True" if prompt.label else "False"
    evaluation = evaluate_next_token(logits, expected, tokenizer, top_k=10)
    return {
        "prediction": evaluation.top1_token,
        "score": int(evaluation.target_rank == 1),
        "target_rank": evaluation.target_rank,
    }

def baseline_for(prompt_id, prompt):
    input_ids, _, _ = activation_by_id[prompt_id]
    with torch.inference_mode():
        logits = causal_lm(input_ids=input_ids, use_cache=False).logits[0, -1]
    return next_token_score(logits, prompt)

def run_patch(source_prompt, target_prompt, k, alpha):
    source_id, target_id = source_prompt.prompt_id, target_prompt.prompt_id
    source_input, source_position, source_activation = activation_by_id[source_id]
    _, target_position, target_activation = activation_by_id[target_id]
    selected_vectors = vectors[:, :k]
    source_coordinates = lens_coordinates(source_activation, selected_vectors)
    target_coordinates_at_position = lens_coordinates(
        target_activation[:, target_position, :], selected_vectors
    )
    target_coordinates = source_coordinates.clone()
    target_coordinates[:, source_position, :] = target_coordinates_at_position
    with torch.inference_mode(), LensCoordinatePatcher(
        model.layers, {LAYER: selected_vectors}, {LAYER: target_coordinates}, alpha=alpha
    ):
        logits = causal_lm(input_ids=source_input, use_cache=False).logits[0, -1]
    return next_token_score(logits, source_prompt)

summary_rows = []
for pair in pairs:
    short, long = pair["short"], pair["long"]
    for direction, source, target in (("long→short", long, short), ("short→long", short, long)):
        baseline = baseline_for(source.prompt_id, source)
        for k in K_VALUES:
            for alpha in ALPHAS:
                intervened = run_patch(source, target, k, alpha)
                summary_rows.append({
                    "problem_id": pair["problem_id"], "direction": direction,
                    "k": k, "alpha": alpha,
                    "baseline_prediction": baseline["prediction"],
                    "baseline_score": baseline["score"],
                    "intervened_prediction": intervened["prediction"],
                    "intervened_score": intervened["score"],
                    "baseline_target_rank": baseline["target_rank"],
                    "intervened_target_rank": intervened["target_rank"],
                })

summary = pd.DataFrame(summary_rows)
display(summary)

## Reading the pilot

`long→short` is supportive if moving long-prompt coefficients toward short-prompt coefficients improves the long baseline. `short→long` is supportive if the reverse move hurts the short baseline. The `alpha=0` rows are an identity check, while changes at `alpha=0.5` and `alpha=1.0` show the direction and strength of the effect.